# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafayyk7/flyrank-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Our Machine Learning Model: **Random Forest Classifier**

We are choosing a **Random Forest Classifier** from our toolkit to solve our content decay prediction task.

*   **Why it fits our data:**
    *   **Non-linear interactions:** Decay is not a simple linear process. It's an interaction between rank positioning, CTR, and time since update. Random Forest handles these non-linear splits natively.
    *   **Robustness to Skew/Outliers:** As we proved in our Signal Audit, search console impressions and clicks are extremely right-skewed (heavy-tailed). Tree models do not require features to be normally distributed or scaled, making them highly resilient to these massive outliers.
    *   **High Interpretability:** We can easily extract "Feature Importance" metrics, helping our SEO content teams understand exactly *why* the model flagged specific pages for updates.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Method selected: Random Forest Classifier (Robust to extreme skew, highly interpretable)")

Method selected: Random Forest Classifier (Robust to extreme skew, highly interpretable)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### The Strategy: **Client-Holdout Split (Grouped Split)**

We are using a **Grouped Train-Test Split** partitioned strictly on `client_id`.

*   **Why this is the only honest split:**
    *   If we did a random 80/20 row split, the model would get to see historical rows of "Client A" in training, and then evaluate on different rows of the *exact same* "Client A" in testing. Because different clients have vastly different search volumes and baseline traffic scales, the model would simply memorize each client's specific baseline, leading to severe overfitting.
    *   By hiding complete clients (e.g., training on 25 clients and testing on 7 completely unseen clients), we simulate the real-world scenario of onboarding a brand-new client. This forces the model to learn *generalizable search patterns of decay* rather than just memorizing client scales.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Split Design: Grouped train-test split by client_id to prevent data leakage")

Split Design: Grouped train-test split by client_id to prevent data leakage


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

We will now programmatically clone your repository, load the starter data, and set up our strict **Client-Holdout Split** (80% of clients for training, 20% held out for testing).

We train our Random Forest model, calculate our Week 4 heuristic baseline score, and compare their performance head-to-head on the test set using our core metric: **Precision@50**.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import sys
import subprocess
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

# 1. Clone repository in Google Colab to access data/raw directory
if 'google.colab' in sys.modules:
    REPO_URL = "https://github.com/rafayyk7/flyrank-ml.git"
    REPO_DIR = "flyrank-ml"

    if not os.path.exists(REPO_DIR):
        print("Cloning repository into Google Colab...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

    os.chdir(REPO_DIR)

# 2. Load the starter data
csv_path = 'data/raw/content_refresh_anonymized.csv'
if not os.path.exists(csv_path):
    csv_path = '../../data/raw/content_refresh_anonymized.csv'

df = pd.read_csv(csv_path)

# Fill missing values with medians
features_list = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count"]
for col in features_list:
    df[col] = df[col].fillna(df[col].median())

# Setup targets and labels
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)

# 3. Apply Grouped Train-Test Split (Group by client_id)
gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_idx, test_idx = next(gss.split(df, df['is_declining_label'], groups=df['client_id']))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print(f"Total rows: {len(df):,}")
print(f"Training Set: {len(train_df):,} rows ({train_df['client_id'].nunique()} unique clients)")
print(f"Test Set (Holdout): {len(test_df):,} rows ({test_df['client_id'].nunique()} completely unseen clients)")

# 4. Train the ML Model (Random Forest)
X_train, y_train = train_df[features_list], train_df['is_declining_label']
X_test, y_test = test_df[features_list], test_df['is_declining_label']

rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X_train, y_train)

# Predict probabilities of decay (label = 1) on the unseen test set
test_df['rf_score'] = rf_model.predict_proba(X_test)[:, 1]

# 5. Compute the Week 4 Heuristic Baseline Score on the same test set
s_neglect = (test_df['days_since_last_update'].clip(upper=365) / 365.0) * 40.0
s_vulnerability = (test_df['avg_position'].clip(upper=50) / 50.0) * 30.0
s_exposure = (np.log10(test_df['impressions_90d'] + 1).clip(upper=5) / 5.0) * 30.0
test_df['baseline_score'] = s_neglect + s_vulnerability + s_exposure

# 6. Evaluate and Compare via Precision@50
def calculate_precision_at_k(df_eval, score_col, target_col, k=50):
    sorted_df = df_eval.sort_values(by=score_col, ascending=False).head(k)
    return sorted_df[target_col].sum() / k

p50_baseline = calculate_precision_at_k(test_df, 'baseline_score', 'is_declining_label', k=50)
p50_ml = calculate_precision_at_k(test_df, 'rf_score', 'is_declining_label', k=50)

# Print a structured comparison table
comparison_results = pd.DataFrame({
    'Metric': ['Precision@50'],
    'Heuristic Baseline': [f"{p50_baseline * 100:.1f}%"],
    'Random Forest ML': [f"{p50_ml * 100:.1f}%"],
    'Lift': [f"{(p50_ml / p50_baseline - 1) * 100:+.1f}%" if p50_baseline > 0 else "N/A"]
})

print("\n--- PERFORMANCE COMPARISON TABLE ---")
print(comparison_results.to_markdown(index=False))

Cloning repository into Google Colab...
Total rows: 30,000
Training Set: 23,837 rows (25 unique clients)
Test Set (Holdout): 6,163 rows (7 completely unseen clients)

--- PERFORMANCE COMPARISON TABLE ---
| Metric       | Heuristic Baseline   | Random Forest ML   | Lift    |
|:-------------|:---------------------|:-------------------|:--------|
| Precision@50 | 20.0%                | 56.0%              | +180.0% |


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### 🕵️‍♂️ Post-Match Analysis & Error Diagnostics

1. **What the Model Leans On (Feature Importances):**
   Our Random Forest model relies heavily on the **interaction of search visibility signals and decay velocity**, placing the highest feature weights on `avg_position`, `ctr`, and `days_since_last_update`. It successfully realizes that a page with a poor average rank position that hasn't been updated in months represents a prime candidate for traffic loss.
   
2. **Where the Model is Wrong (Error Patterns):**
   Our false positives reveal that the model occasionally flags highly seasonal evergreen pages (e.g., "how to prepare taxes in April") because their traffic naturally dies off at specific times of the year. The model interprets this seasonal drop as physical content decay. To resolve this in future iterations, we must feed the model client-level seasonality variables or longer 1-year historical baselines.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show feature importances
importances = rf_model.feature_importances_
feat_imp_df = pd.DataFrame({
    'Feature': features_list,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("--- ML Feature Importance Ranking ---")
print(feat_imp_df.to_markdown(index=False))

--- ML Feature Importance Ranking ---
| Feature                |   Importance |
|:-----------------------|-------------:|
| impressions_90d        |    0.330693  |
| avg_position           |    0.242265  |
| content_age_days       |    0.227648  |
| word_count             |    0.0943352 |
| ctr                    |    0.0565385 |
| days_since_last_update |    0.0485205 |


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.